<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/hem_finetune/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets==3.6.0 transformers torch evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import warnings
import evaluate
import pandas as pd
import numpy as np
import os
import re
import pickle
import gc
import torch
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from tqdm import tqdm
from google.colab import drive
from transformers import EarlyStoppingCallback

drive.mount('/content/drive')

with open("/content/drive/MyDrive/datasets/balanced_wrong_predictions.pkl", "rb") as f:
    df = pickle.load(f)

Mounted at /content/drive


In [3]:
print(df['label'].value_counts())

label
0    164884
1    164884
Name: count, dtype: int64


In [4]:
# early stopping by overriding eval_f1 and eval_loss metrics to prevent over fitting
class DualMetricEarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3, min_delta_f1=1e-7, min_delta_loss=1e-7, tolerance_f1=0.02, tolerance_loss=0.02):
        self.patience = patience
        self.counter = 0
        self.best_f1 = None
        self.best_loss = None
        self.min_delta_f1 = min_delta_f1
        self.min_delta_loss = min_delta_loss
        self.tolerance_f1 = tolerance_f1
        self.tolerance_loss = tolerance_loss

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_f1 = metrics.get("eval_f1")
        current_loss = metrics.get("eval_loss")

        if current_f1 is None or current_loss is None:
            return control

        if self.best_f1 is None or self.best_loss is None:
            self.best_f1 = current_f1
            self.best_loss = current_loss
            self.counter = 0
        else:
            f1_improved = current_f1 > self.best_f1 + self.min_delta_f1
            f1_decline_ok = current_f1 >= self.best_f1 - self.tolerance_f1

            loss_improved = current_loss < self.best_loss - self.min_delta_loss
            loss_decline_ok = current_loss <= self.best_loss + self.tolerance_loss

            if f1_improved or loss_improved or (f1_decline_ok and loss_decline_ok):
                self.best_f1 = max(self.best_f1, current_f1)
                self.best_loss = min(self.best_loss, current_loss)
                self.counter = 0
            else:
                self.counter += 1
                print(f"[EarlyStopping] No acceptable improvement. Patience {self.counter}/{self.patience}")

        if self.counter >= self.patience:
            print(f"[EarlyStopping] Triggered at epoch {state.epoch}.")
            control.should_training_stop = True

        return control

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    preds = logits.argmax(dim=1).to(device)
    labels = labels.to(device)

    num_classes = torch.max(labels).item() + 1
    f1_total = 0.0
    precision_total = 0.0
    recall_total = 0.0

    for cls in range(num_classes):
        tp = ((preds == cls) & (labels == cls)).sum()
        fp = ((preds == cls) & (labels != cls)).sum()
        fn = ((preds != cls) & (labels == cls)).sum()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        precision_total += precision
        recall_total += recall
        f1_total += f1

    macro_precision = precision_total / num_classes
    macro_recall = recall_total / num_classes
    macro_f1 = f1_total / num_classes

    accuracy = (preds == labels).sum().float() / labels.shape[0]

    return {
        "f1": macro_f1.item(),
        "precision": macro_precision.item(),
        "recall": macro_recall.item(),
        "accuracy": accuracy.item()
    }

# which model saved
class PrintBestModelCallback(TrainerCallback):
    def on_train_end(self, args, state, control, **kwargs):
        best_step = getattr(state, "best_step", None)
        best_metric = getattr(state, "best_metric", None)

        if best_step is not None and best_metric is not None:
            print(f"\n[INFO] Best model was at step {best_step} with best {args.metric_for_best_model}: {best_metric}")
        else:
            print("\n[INFO] Best step or metric not available (possibly due to resumed checkpoint).")


# split dataset
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.12, random_state=63, stratify=df['label'])

# tokenization for huggingfaceapi
train_df = {"text": list(X_train), "label": list(y_train)}
test_df = {"text": list(X_test), "label": list(y_test)}
train_dataset = Dataset.from_dict(train_df)
eval_dataset = Dataset.from_dict(test_df)
model_path = "/content/drive/MyDrive/models/gp_model_24750"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

#load model
model = AutoModelForSequenceClassification.from_pretrained(model_path)

train_batch_size = 64
num_epochs = 4
num_train_steps = (len(train_dataset) // train_batch_size) * num_epochs
warmup_steps = int(0.05 * num_train_steps)

# define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/models/gp_model_finetuned",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    max_steps=5000,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=None,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=1,
    fp16=True,
)

# define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks = [DualMetricEarlyStoppingCallback(
    patience=14,
    min_delta_f1=1e-7,
    min_delta_loss=1e-7,
    tolerance_f1=0.05,
    tolerance_loss=0.03
    ),
                 PrintBestModelCallback()]
)

# training model
trainer.train(resume_from_checkpoint=True)

# saving model
save_path = "/content/drive/MyDrive/models/gp_modelv2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Map:   0%|          | 0/290195 [00:00<?, ? examples/s]

Map:   0%|          | 0/39573 [00:00<?, ? examples/s]

<ipython-input-4-21c39cfb8ad8>:155: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss



[INFO] Best step or metric not available (possibly due to resumed checkpoint).


('/content/drive/MyDrive/models/gp_modelv2/tokenizer_config.json',
 '/content/drive/MyDrive/models/gp_modelv2/special_tokens_map.json',
 '/content/drive/MyDrive/models/gp_modelv2/spm.model',
 '/content/drive/MyDrive/models/gp_modelv2/added_tokens.json')

In [3]:
# Checkpoint klasörü
checkpoint_path = "/content/drive/MyDrive/models/gp_model_finetuned/checkpoint-4000"

# Temiz model olarak kaydedilecek klasör
final_model_path = "/content/drive/MyDrive/models/gp_model_v2_best"

# Model ve tokenizer'ı yükle
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=False)

# Modeli temiz şekilde kaydet
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Temiz model kaydedildi: {final_model_path}")

Temiz model kaydedildi: /content/drive/MyDrive/models/gp_model_v2_best
